In [ ]:
from textblob import TextBlob

In [ ]:
from google.colab import files
uplodaed = files.upload()

Saving train.csv to train.csv


In [ ]:
import pandas as pd
df= pd.read_csv('/content/train.csv')

In [ ]:
df

,oare_id,transliteration,translation
0,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠI...,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ..."
1,0064939c-59b9-4448-a63d-34612af0a1b5,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr-ilī has received one textile of ordinary ...
2,0073f2c0-524c-4bbf-915a-8c1772a4fb98,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR,... he did not give you a textile. He returned...
3,009fb838-8038-42bc-ad34-5f795b3840ee,KIŠIB šu-(d)EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū..."
4,00aa1c55-c80c-4346-a159-73ad43ab0ff7,um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...,From Šukkutum to Ištar-lamassī and Nitahšušar:...
...,...,...,...
1556,ff3208e4-8ab8-4368-b4df-7b80afa5bc32,um-ma en-nam-a-šur-ma a-na a-la-ḫi-im qí-bi-ma...,From Ennam-Aššur to Ali-ahum: Here 2 men have ...
1557,ff43a284-3d67-4238-8b4a-9b6cb7531e0a,3 ma-na KÙ.BABBAR ṣa-ru-pá-am i-na ší-im SÍG.Ḫ...,Ilī-ašrannī son of Sukkalliya has received 3 m...
1558,ff5747a4-af8a-4100-a906-a2660ae72606,ša-lim-a-šùr a-na a-mur-IŠTAR ú-ṭá-ḫi-ni-a-tí-...,Šalim-Aššur made us approach Amur-Ištar and Ša...
1559,ff777871-97ce-4bfc-bdfb-73352868944d,a-na en-nam-a-šùr qí-bi-ma um-ma IŠTAR-ra-bi₄-...,To Ennam-Aššur from Ištar-rabiʾat: With respec...


In [ ]:
!python -m textblob.download_corpora


[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package conll2000 to /root/nltk_data...
[nltk_data]   Unzipping corpora/conll2000.zip.
[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.
Finished.


In [ ]:
def safe_correct(text):
    blob = TextBlob(text)
    corrected = []
    for word in blob.words:
        if word[0].isupper():
            corrected.append(word)
        else:
            corrected.append(str(TextBlob(word).correct()))
    return " ".join(corrected)

df['translation_corrected'] = df['translation'].apply(
    lambda x: safe_correct(x) if isinstance(x, str) else x
)


KeyboardInterrupt: 

In [ ]:
!wget https://raw.githubusercontent.com/mammothb/symspellpy/master/symspellpy/frequency_dictionary_en_82_765.txt


--2026-01-03 15:23:40--  https://raw.githubusercontent.com/mammothb/symspellpy/master/symspellpy/frequency_dictionary_en_82_765.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1332879 (1.3M) [text/plain]
Saving to: ‘frequency_dictionary_en_82_765.txt’

frequency_dictionar 100%[===================>]   1.27M  --.-KB/s    in 0.03s   

2026-01-03 15:23:40 (39.7 MB/s) - ‘frequency_dictionary_en_82_765.txt’ saved [1332879/1332879]



In [ ]:
!pip install symspellpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.6/159.6 kB 12.3 MB/s eta 0:00:00


In [ ]:
from symspellpy import SymSpell, Verbosity
import re

sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)

dictionary_path = "/usr/share/dict/words"  # fallback
# Better: use symspell frequency dictionary
sym_spell.load_dictionary(
    "frequency_dictionary_en_82_765.txt",  # download once
    term_index=0,
    count_index=1
)


True

In [ ]:
def symspell_safe_correct(text):
    if not isinstance(text, str):
        return text

    words = re.findall(r"\b\w+\b", text)
    corrected_words = []

    for word in words:
        # Skip proper nouns
        if word[0].isupper():
            corrected_words.append(word)
            continue

        suggestions = sym_spell.lookup(
            word,
            Verbosity.CLOSEST,
            max_edit_distance=2
        )

        corrected_words.append(
            suggestions[0].term if suggestions else word
        )

    return " ".join(corrected_words)


In [ ]:
df['translation_corrected'] = df['translation'].apply(symspell_safe_correct)


In [ ]:
df[['translation', 'translation_corrected']].head(10)


,translation,translation_corrected
0,"Seal of Mannum-balum-Aššur son of Ṣilli-Adad, ...",Seal of Mannum balm Aššur son of Ṣilli Adad se...
1,Itūr-ilī has received one textile of ordinary ...,Itūr ill has received one textile of ordinary ...
2,... he did not give you a textile. He returned...,he did not give you a textile He returned and ...
3,"Seal of Šu-Illil son of Šu-Kūbum, seal of Ṣilū...",Seal of Šu Illil son of Šu Kūbum seal of Ṣilūl...
4,From Šukkutum to Ištar-lamassī and Nitahšušar:...,From Šukkutum to Ištar lamas and Nitahšušar Wh...
5,"From Šu-Tammuzī, Elaya, Ennam-Aššur and Lamass...",From Šu Tammuzī Elaya Ennam Aššur and Lamassī ...
6,"Seal of Ali-ahum, seal of Ali-ilī, seal of Abi...",Seal of Ali hum seal of Ali ill seal of Abila ...
7,We entered Šalim-Aššur's house to settle the c...,We entered Šalim Aššur a house to settle the c...
8,A tablet about 10 minas of silver of Puzur-Ass...,A tablet about of minds of silver of Puzur Ass...
9,"17 shekels of silver, the price of a donkey, 3...",of shekels of silver the price of a donkey of ...


In [ ]:
df.drop(columns=['translation'], inplace=True)

In [ ]:
df.rename(columns={'translation_corrected': 'translation'}, inplace=True)

In [ ]:
df

,oare_id,transliteration,translation
0,004a7dbd-57ce-46f8-9691-409be61c676e,KIŠIB ma-nu-ba-lúm-a-šur DUMU ṣí-lá-(d)IM KIŠI...,Seal of Mannum balm Aššur son of Ṣilli Adad se...
1,0064939c-59b9-4448-a63d-34612af0a1b5,1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé,Itūr ill has received one textile of ordinary ...
2,0073f2c0-524c-4bbf-915a-8c1772a4fb98,TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR,he did not give you a textile He returned and ...
3,009fb838-8038-42bc-ad34-5f795b3840ee,KIŠIB šu-(d)EN.LÍL DUMU šu-ku-bi-im KIŠIB ṣí-l...,Seal of Šu Illil son of Šu Kūbum seal of Ṣilūl...
4,00aa1c55-c80c-4346-a159-73ad43ab0ff7,um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...,From Šukkutum to Ištar lamas and Nitahšušar Wh...
...,...,...,...
1556,ff3208e4-8ab8-4368-b4df-7b80afa5bc32,um-ma en-nam-a-šur-ma a-na a-la-ḫi-im qí-bi-ma...,From Ennam Aššur to Ali hum Here a men have re...
1557,ff43a284-3d67-4238-8b4a-9b6cb7531e0a,3 ma-na KÙ.BABBAR ṣa-ru-pá-am i-na ší-im SÍG.Ḫ...,Ilī ašrannī son of Sukkalliya has received a m...
1558,ff5747a4-af8a-4100-a906-a2660ae72606,ša-lim-a-šùr a-na a-mur-IŠTAR ú-ṭá-ḫi-ni-a-tí-...,Šalim Aššur made us approach Amur Ištar and Ša...
1559,ff777871-97ce-4bfc-bdfb-73352868944d,a-na en-nam-a-šùr qí-bi-ma um-ma IŠTAR-ra-bi₄-...,To Ennam Aššur from Ištar habitat With respect...


In [ ]:
df.to_csv('corrected_translations.csv', index=False)